In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA

sns.set_style("whitegrid")

In [ ]:
# task 1

In [ ]:
titanic = sns.load_dataset("titanic")

titanic.shape

In [ ]:
titanic.head()

In [ ]:
titanic.dtypes

In [ ]:
missing = titanic.isnull().sum()
missing_percent = (missing / len(titanic)) * 100

pd.DataFrame({
    "Missing Values": missing,
    "Percentage (%)": missing_percent
})

In [ ]:
titanic["survived"].value_counts(normalize=True)

In [ ]:
plt.figure()
sns.histplot(titanic["age"], kde=True)
plt.title("Age Distribution")
plt.show()

In [ ]:
plt.figure()
sns.barplot(x="sex", y="survived", data=titanic)
plt.title("Survival Rate by Sex")
plt.show()

The dataset contains both numerical and categorical variables. Numerical features include age, fare, and sibsp, while categorical features include sex, class, and embarked. Some columns have missing values, especially age and deck, where deck has many missing entries.

The target variable is survived, and the data is slightly imbalanced, with more people not surviving than surviving. The survival rate is around 38%.

The age distribution shows that most passengers were young adults, but there are also children and older people. The survival analysis shows that females had a much higher survival rate than males, which suggests that gender played an important role in survival.

In [ ]:
# task 2

In [ ]:
# Handle missing Values
# Fill missing age
titanic["age"] = titanic["age"].fillna(titanic["age"].median())

# Fill missing embarked
titanic["embarked"] = titanic["embarked"].fillna(titanic["embarked"].mode()[0])

In [ ]:
# Encode categorical features
# Label encoding for sex (male=1, female=0)
titanic["sex"] = titanic["sex"].map({"male": 1, "female": 0})

# One-hot encoding for embarked
titanic = pd.get_dummies(titanic, columns=["embarked"], drop_first=True)

In [ ]:
# Create new features
# Family size
titanic["family_size"] = titanic["sibsp"] + titanic["parch"] + 1

# Is alone
titanic["is_alone"] = (titanic["family_size"] == 1).astype(int)

In [ ]:
titanic.drop(columns=[
    "alive", 
    "who", 
    "adult_male", 
    "embark_town", 
    "class", 
    "alone"
], inplace=True)

In [ ]:
titanic.head()
titanic.dtypes

In this step, the dataset was cleaned and transformed for machine learning. Missing values in age were filled using the median, and missing values in embarked were filled using the most frequent value. The deck column was removed because it had too many missing values.

Categorical variables were converted into numeric format. The sex column was label encoded, and embarked was one-hot encoded to create new binary columns.

New features were also created. The family_size feature shows how many family members each passenger had, and is_alone indicates whether the passenger was traveling alone.

Finally, unnecessary and non-numeric columns were removed. The resulting dataset now contains only numeric features and is ready for machine learning models.

In [ ]:
# task 3

In [ ]:
# Select numeric features for scaling
numeric_features = ['age', 'fare', 'family_size']
data_to_scale = titanic[numeric_features]

# Apply StandardScaler
std_scaler = StandardScaler()
data_std = std_scaler.fit_transform(data_to_scale)

# Apply MinMaxScaler
minmax_scaler = MinMaxScaler()
data_minmax = minmax_scaler.fit_transform(data_to_scale)

# Visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Age distributions
sns.histplot(titanic['age'], kde=True, ax=axes[0,0], color='blue').set_title('Original Age')
sns.histplot(data_std[:,0], kde=True, ax=axes[0,1], color='green').set_title('StandardScaler Age')
sns.histplot(data_minmax[:,0], kde=True, ax=axes[0,2], color='orange').set_title('MinMaxScaler Age')

# Fare distributions
sns.histplot(titanic['fare'], kde=True, ax=axes[1,0], color='blue').set_title('Original Fare')
sns.histplot(data_std[:,1], kde=True, ax=axes[1,1], color='green').set_title('StandardScaler Fare')
sns.histplot(data_minmax[:,1], kde=True, ax=axes[1,2], color='orange').set_title('MinMaxScaler Fare')

plt.tight_layout()
plt.show()

StandardScaler: It centers the data around a mean of 0 and scales it based on standard deviation. This is useful for algorithms like PCA or SVM, but it doesn't bound the values to a specific range.

MinMaxScaler: It squashes all values strictly between 0 and 1. This is highly sensitive to outliers (like the high Fares in Titanic), as a single extreme value can "compress" the rest of the data into a very small range.

In [ ]:
# Compute correlation matrix for numeric columns only
plt.figure(figsize=(12, 10))
corr_matrix = titanic.corr(numeric_only=True)

# Plotting the heatmap
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Feature Correlation Matrix")
plt.show()

# Display correlations with the target variable 'survived'
print("Correlation with Survived:")
print(corr_matrix['survived'].sort_values(ascending=False))

We select features that have an absolute correlation value of at least 0.1 with the survived column.

In [ ]:
# Identify features with absolute correlation >= 0.1
threshold = 0.1
relevant_features = corr_matrix.index[corr_matrix['survived'].abs() >= threshold].tolist()

# Remove the target variable itself from the list
if 'survived' in relevant_features:
    relevant_features.remove('survived')

print(f"Selected Features: {relevant_features}")

This step reduces the dimensionality of your data while trying to keep as much information (variance) as possible.

In [ ]:
# Prepare data for PCA (using the selected features)
pca_data = titanic[relevant_features]

# PCA requires data to be standardized first
pca_scaled = StandardScaler().fit_transform(pca_data)

# Apply PCA to reduce to 2 dimensions
pca = PCA(n_components=2)
pca_components = pca.fit_transform(pca_scaled)

# Create a DataFrame for the PCA results
pca_df = pd.DataFrame(data=pca_components, columns=['PC1', 'PC2'])
pca_df['survived'] = titanic['survived'].values

print(f"Total Explained Variance: {pca.explained_variance_ratio_.sum():.2f}")

# Visualize PCA results
plt.figure(figsize=(10, 7))
sns.scatterplot(x='PC1', y='PC2', hue='survived', data=pca_df, palette='viridis', alpha=0.7)
plt.title('PCA: 2D Projection of Titanic Dataset')
plt.show()

In [ ]:
# task 4

In [ ]:
# --- STEP 0: Safety Check (Fixing the Error) ---
# Ensure we only use numeric columns for PCA. 
# This removes any leftover strings like 'A', 'C', 'male', etc.
X = titanic.drop(columns=['survived'], errors='ignore')
X_numeric = X.select_dtypes(include=[np.number])

print(f"Features used for PCA: {X_numeric.columns.tolist()}")

# --- STEP 1: Standardize all numeric features ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_numeric)

# --- STEP 2: Fit PCA with all components ---
pca_all = PCA()
pca_all.fit(X_scaled)

# Plot Explained Variance Ratio
exp_var = pca_all.explained_variance_ratio_
cum_var = np.cumsum(exp_var)

plt.figure(figsize=(10, 5))
plt.bar(range(1, len(exp_var) + 1), exp_var, alpha=0.5, align='center', label='Individual Variance')
plt.step(range(1, len(cum_var) + 1), cum_var, where='mid', label='Cumulative Variance', color='red')
plt.axhline(y=0.8, color='green', linestyle='--', label='80% Variance Threshold')
plt.xlabel('Principal Component Index')
plt.ylabel('Explained Variance Ratio')
plt.title('PCA Explained Variance')
plt.legend()
plt.show()

# Calculate how many components for 80%
n_80 = np.argmax(cum_var >= 0.8) + 1
print(f"Number of components needed for 80% variance: {n_80}")

# --- STEP 3: Refit PCA with 2 components ---
pca_2 = PCA(n_components=2)
X_pca_2 = pca_2.fit_transform(X_scaled)

# Create a DF for visualization
pca_df = pd.DataFrame(X_pca_2, columns=['PC1', 'PC2'])
pca_df['Survived'] = titanic['survived'].values

# Plot the 2D Projection
plt.figure(figsize=(10, 7))
sns.scatterplot(x='PC1', y='PC2', hue='Survived', data=pca_df, palette='coolwarm', alpha=0.8)
plt.title('PCA 2D Projection: Survivors vs Non-Survivors')
plt.show()

1. Does PCA help separate the classes?
In the scatter plot, you will notice some grouping (e.g., survivors might cluster on one side of PC1), but there is usually significant overlap. While PCA captures the main directions of variance, the "Survival" outcome depends on complex interactions (like "Women and Children first") that a 2D linear projection might not perfectly separate.

2. Would you use PCA as a preprocessing step here?

Recommendation: No, for this specific dataset, the original feature space is likely sufficient.

Reasoning:

Small Feature Set: We only have about 10-12 features. PCA is most useful when you have hundreds of features (e.g., images or gene data).

Interpretability: In Titanic, we care why someone survived (e.g., being in 1st class or being female). PCA creates "Principal Components" which are hard to explain to a human.

Information Loss: As seen in the variance plot, 2 components capture much less than 100% of the data. Keeping the original features preserves all the information for the model.